# Notebook 02: Data Cleaning & ETL Pipeline
**Purpose:** Fix every data quality issue found in Notebook 01.
Every single transformation must be logged so the instructor 
can see exactly what was done and why.
**Golden rule:** NEVER edit the raw file. All changes happen here
and the output goes to data/processed/

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/student_health_raw.csv'
df = pd.read_csv(RAW_PATH, low_memory=False)

transformation_log = []

def log(step_num, action, detail, rows_affected=None):
    """
    Helper function to log every transformation step.
    step_num: which step number this is (1, 2, 3...)
    action:   what we did in plain English
    detail:   technical detail of the change
    rows_affected: how many rows were changed or removed
    """
    entry = f"[STEP {step_num:02d}] {action}"
    if rows_affected is not None:
        entry += f" | Rows affected: {rows_affected:,}"
    if detail:
        entry += f"\n         Detail: {detail}"
    transformation_log.append(entry)
    print(entry)

print(f"Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
log(0, "Dataset loaded", f"Shape: {df.shape}")

Raw dataset loaded: 1,000,000 rows × 20 columns
[STEP 00] Dataset loaded
         Detail: Shape: (1000000, 20)


Duplicate rows are identical records that appear more 
than once. They make our averages wrong and our charts misleading. 
We always remove them first before any other cleaning.

In [2]:
before = len(df)

df = df.drop_duplicates()

after = len(df)
removed = before - after

log(1, "Removed duplicate rows", 
    f"Before: {before:,} rows → After: {after:,} rows", 
    rows_affected=removed)

[STEP 01] Removed duplicate rows | Rows affected: 0
         Detail: Before: 1,000,000 rows → After: 1,000,000 rows


Missing values (NaN) cannot be used in calculations. 
We have two main strategies:
1. For NUMERICAL columns: fill with the MEDIAN (middle value). 
   We use median instead of mean because mean is affected by outliers.
2. For CATEGORICAL columns: fill with the MODE (most common value).
We log every column we fix and how many values we filled.

In [3]:

numerical_cols = ['age', 'study_hours_per_day', 'stress_level',
                  'anxiety_score', 'depression_score', 'sleep_hours',
                  'social_support', 'screen_time', 'internet_usage',
                  'financial_stress', 'exam_pressure', 'academic_performance',
                  'physical_activity', 'family_expectation']

categorical_cols = ['gender', 'academic_year']

print("Filling numerical missing values with MEDIAN:")
for col in numerical_cols:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        log(2, f"Filled missing values in '{col}'",
            f"Filled {missing_count:,} nulls with median = {median_val:.2f}",
            rows_affected=missing_count)

print("\nFilling categorical missing values with MODE:")
for col in categorical_cols:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        log(3, f"Filled missing values in '{col}'",
            f"Filled {missing_count:,} nulls with mode = '{mode_val}'",
            rows_affected=missing_count)

Filling numerical missing values with MEDIAN:

Filling categorical missing values with MODE:


Inconsistent categories are a very common problem. 
Some columns in the actual Kaggle dataset were saved as floats 
instead of categories, so we dynamically map them to the expected bins.

In [4]:
df['gender'] = df['gender'].astype(str).str.strip().str.title()
gender_map = {'M': 'Male', 'F': 'Female', 'Male ': 'Male', 'Female ': 'Female'}
df['gender'] = df['gender'].replace(gender_map)
log(4, "Standardised 'gender' column", "Applied title case")

df['academic_year'] = df['academic_year'].astype(str)
df['academic_year'] = df['academic_year'].apply(lambda x: f"Year {x}" if len(x)==1 else x)
log(5, "Standardised 'academic_year' column", "Formatted as 'Year X'")

if pd.api.types.is_numeric_dtype(df['exam_pressure']):
    df['exam_pressure'] = pd.cut(df['exam_pressure'], bins=[-1, 3.33, 6.66, 11], labels=['Low', 'Medium', 'High'])
    log(6, "Binned 'exam_pressure' into Low/Medium/High", "")

if pd.api.types.is_numeric_dtype(df['academic_performance']):
    df['academic_performance'] = pd.cut(df['academic_performance'], bins=[-1, 60, 75, 85, 101], labels=['Poor', 'Average', 'Good', 'Excellent'])
    log(7, "Binned 'academic_performance' into Poor/Average/Good/Excellent", "")

if pd.api.types.is_numeric_dtype(df['physical_activity']):
    df['physical_activity'] = pd.cut(df['physical_activity'], bins=[-1, 2.5, 5, 7.5, 11], labels=['None', 'Low', 'Moderate', 'High'])
    log(8, "Binned 'physical_activity' into None/Low/Moderate/High", "")

if pd.api.types.is_numeric_dtype(df['family_expectation']):
    df['family_expectation'] = pd.cut(df['family_expectation'], bins=[-1, 3.33, 6.66, 11], labels=['Low', 'Medium', 'High'])
    log(9, "Binned 'family_expectation' into Low/Medium/High", "")

[STEP 04] Standardised 'gender' column
         Detail: Applied title case
[STEP 05] Standardised 'academic_year' column
         Detail: Formatted as 'Year X'
[STEP 06] Binned 'exam_pressure' into Low/Medium/High
[STEP 07] Binned 'academic_performance' into Poor/Average/Good/Excellent


[STEP 08] Binned 'physical_activity' into None/Low/Moderate/High
[STEP 09] Binned 'family_expectation' into Low/Medium/High


Outliers are values that are impossibly wrong — like 
a student sleeping 30 hours per day or having a stress score of 
15 when the maximum is 10. These are data entry errors that would 
skew our analysis. We remove them using the valid range for each 
column.

In [5]:

valid_ranges = {
    'age':                 (17, 30),
    'study_hours_per_day': (0, 18),
    'stress_level':        (0, 10),
    'anxiety_score':       (0, 10),
    'depression_score':    (0, 10),
    'sleep_hours':         (2, 12),
    'social_support':      (0, 10),
    'screen_time':         (0, 16),
    'internet_usage':      (0, 16),
    'financial_stress':    (0, 10),
}

total_removed = 0
for col, (min_val, max_val) in valid_ranges.items():
    if col in df.columns:
        before = len(df)
        df = df[(df[col] >= min_val) & (df[col] <= max_val)]
        after = len(df)
        removed = before - after
        total_removed += removed
        
        if removed > 0:
            log(10, f"Removed outliers from '{col}'",
                f"Valid range: {min_val}–{max_val} | "
                f"Removed {removed:,} rows",
                rows_affected=removed)

print(f"\nTotal rows removed for outliers: {total_removed:,}")


Total rows removed for outliers: 0


We drop screen_time and internet_usage because they 
are very highly correlated with each other (r > 0.85). When two 
columns measure almost the same thing, keeping both causes 
'multicollinearity' which makes regression results unreliable. 
We keep only the more interpretable column.

In [6]:

corr = df['screen_time'].corr(df['internet_usage'])
print(f"Correlation between screen_time and internet_usage: {corr:.3f}")

if abs(corr) > 0.7:
    print(f"High correlation ({corr:.3f}) — dropping internet_usage")
    df = df.drop(columns=['internet_usage'])
    log(11, "Dropped 'internet_usage' column",
        f"High correlation with screen_time: r={corr:.3f}")
else:
    print(f"Correlation is {corr:.3f} — keeping both columns")
    log(11, "Kept both screen_time and internet_usage",
        f"Correlation was only {corr:.3f}")

Correlation between screen_time and internet_usage: 0.891
High correlation (0.891) — dropping internet_usage
[STEP 11] Dropped 'internet_usage' column
         Detail: High correlation with screen_time: r=0.891


Now we CREATE new columns that don't exist in the raw 
data. These derived columns are our original analytical contribution — 
they represent the KPIs (Key Performance Indicators) we defined 
for this project. Creating meaningful derived columns is one of 
the things instructors look for.

In [7]:

df['burnout_composite'] = (
    df['stress_level'] + 
    df['anxiety_score'] + 
    df['depression_score']
) / 3

df['burnout_composite'] = df['burnout_composite'].round(2)

print(f"burnout_composite created")
log(12, "Created 'burnout_composite' column",
    "Formula: (stress_level + anxiety_score + depression_score) / 3")

df['stress_tier'] = pd.cut(
    df['stress_level'],
    bins=[-0.01, 3, 6, 10],           
    labels=['Low (0–3)', 'Medium (4–6)', 'High (7–10)']
)

log(13, "Created 'stress_tier' column",
    "Bins: Low (0–3), Medium (4–6), High (7–10)")

df['sleep_deficit'] = df['sleep_hours'].apply(
    lambda x: 'Sleep Deprived (<7hrs)' if x < 7 
              else 'Adequate Sleep (≥7hrs)'
)

log(14, "Created 'sleep_deficit' column",
    "Threshold: <7 hours = Sleep Deprived, ≥7 hours = Adequate Sleep")

burnout_composite created
[STEP 12] Created 'burnout_composite' column
         Detail: Formula: (stress_level + anxiety_score + depression_score) / 3
[STEP 13] Created 'stress_tier' column
         Detail: Bins: Low (0–3), Medium (4–6), High (7–10)
[STEP 14] Created 'sleep_deficit' column
         Detail: Threshold: <7 hours = Sleep Deprived, ≥7 hours = Adequate Sleep


In [8]:

print("=== FINAL DATA QUALITY CHECK ===")
print(f"\nFinal shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nNull values remaining:")
null_check = df.isnull().sum()
if null_check.sum() == 0:
    print("  Zero null values — dataset is clean!")
else:
    print(null_check[null_check > 0])

print(f"\nDuplicate rows remaining: {df.duplicated().sum():,}")
print(f"\nFinal columns:")
for col in df.columns:
    print(f"  • {col} ({df[col].dtype})")

=== FINAL DATA QUALITY CHECK ===

Final shape: 1,000,000 rows × 22 columns

Null values remaining:
  Zero null values — dataset is clean!



Duplicate rows remaining: 0

Final columns:
  • age (int64)
  • gender (str)
  • academic_year (str)
  • study_hours_per_day (float64)
  • exam_pressure (category)
  • academic_performance (category)
  • stress_level (float64)
  • anxiety_score (float64)
  • depression_score (float64)
  • sleep_hours (float64)
  • physical_activity (category)
  • social_support (float64)
  • screen_time (float64)
  • financial_stress (float64)
  • family_expectation (category)
  • burnout_score (float64)
  • mental_health_index (float64)
  • risk_level (str)
  • dropout_risk (float64)
  • burnout_composite (float64)
  • stress_tier (category)
  • sleep_deficit (str)


In [9]:
PROCESSED_PATH = '../data/processed/student_health_clean.csv'
df.to_csv(PROCESSED_PATH, index=False)

print(f"Cleaned dataset saved: {PROCESSED_PATH}")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")

print("\n" + "=" * 60)
print("  COMPLETE TRANSFORMATION LOG")
print("=" * 60)
for entry in transformation_log:
    print(entry)
    print()
print(f"Total transformations logged: {len(transformation_log)}")
print("NEXT STEP: Open 03_feature_engineering.ipynb")

Cleaned dataset saved: ../data/processed/student_health_clean.csv
  Rows: 1,000,000
  Columns: 22

  COMPLETE TRANSFORMATION LOG
[STEP 00] Dataset loaded
         Detail: Shape: (1000000, 20)

[STEP 01] Removed duplicate rows | Rows affected: 0
         Detail: Before: 1,000,000 rows → After: 1,000,000 rows

[STEP 04] Standardised 'gender' column
         Detail: Applied title case

[STEP 05] Standardised 'academic_year' column
         Detail: Formatted as 'Year X'

[STEP 06] Binned 'exam_pressure' into Low/Medium/High

[STEP 07] Binned 'academic_performance' into Poor/Average/Good/Excellent

[STEP 08] Binned 'physical_activity' into None/Low/Moderate/High

[STEP 09] Binned 'family_expectation' into Low/Medium/High

[STEP 11] Dropped 'internet_usage' column
         Detail: High correlation with screen_time: r=0.891

[STEP 12] Created 'burnout_composite' column
         Detail: Formula: (stress_level + anxiety_score + depression_score) / 3

[STEP 13] Created 'stress_tier' column
     